In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import copy
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error
from tqdm import tqdm

# ==========================================
# 1. 实验配置 (针对 Junyi 数据集)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 50
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.2

        # 训练参数
        self.batch_size = 128
        self.epochs = 5
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "junyi"

        # 消融实验控制开关
        self.use_student_interaction = True
        self.use_knowledge_concept = True

# ==========================================
# 2. 数据处理模块 (增强版 PSLC 适配)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]

        seq_len = len(q_seq)
        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        try:
            import EduData
            return True
        except ImportError:
            print("正在尝试自动安装 EduData 库...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                return True
            except:
                return False

    def download_data(self):
        print(f"正在准备下载/检查数据集: {self.config.dataset_name}")
        if self._install_edudata():
            from EduData import get_data
            get_data(self.config.dataset_name, self.config.data_dir)

        for root, dirs, files in os.walk(self.config.data_dir):
            for file in files:
                if 'junyi' in root.lower() and ('log' in file.lower() or 'data' in file.lower() or 'pslc' in file.lower()):
                    if file.endswith('.csv') or file.endswith('.txt'):
                        return os.path.join(root, file)
        return None

    def process(self):
        file_path = self.download_data()
        if not file_path:
            raise FileNotFoundError("未找到 Junyi 有效数据文件。")

        print(f"读取 Junyi 数据: {file_path}")
        try:
            df = pd.read_csv(file_path, sep='\t', engine='python', nrows=500000)
            if df.shape[1] <= 1:
                df = pd.read_csv(file_path, sep=None, engine='python', nrows=500000)
        except:
            df = pd.read_csv(file_path, sep=None, engine='python', nrows=500000)

        cols_lower = {c.lower(): c for c in df.columns}

        user_col = None
        for k in ['anon student id', 'user_id', 'uid', 'student_id']:
            if k in cols_lower:
                user_col = cols_lower[k]
                break

        item_col = None
        for k in ['problem name', 'exercise', 'problem_id', 'item_id', 'problem_name']:
            if k in cols_lower:
                item_col = cols_lower[k]
                break

        correct_col = None
        for k in ['outcome', 'correct', 'is_correct', 'score']:
            if k in cols_lower:
                correct_col = cols_lower[k]
                break

        if not (user_col and item_col and correct_col):
            print(f"当前文件列名: {list(df.columns)}")
            raise ValueError("无法匹配 Junyi 数据集列名")

        df = df[[user_col, item_col, correct_col]].dropna()

        if df[correct_col].dtype == object:
            df[correct_col] = df[correct_col].str.upper().map({'CORRECT': 1, 'INCORRECT': 0, 'HINT': 0}).fillna(0)
        else:
            df[correct_col] = df[correct_col].apply(lambda x: 1 if x >= 0.5 else 0)

        user_map = {val: i+1 for i, val in enumerate(df[user_col].unique())}
        item_map = {val: i+1 for i, val in enumerate(df[item_col].unique())}
        df['uid'] = df[user_col].map(user_map)
        df['iid'] = df[item_col].map(item_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(item_map) + 1
        self.config.num_concepts = self.config.num_questions

        print(f"统计: 学生 {self.config.num_students}, 题目 {self.config.num_questions}")

        item_diff = df.groupby('iid')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_diff.items():
            diff_values[iid] = diff

        q_k_relation = np.eye(self.config.num_questions)

        all_q, all_r, all_s = [], [], []
        users = df['uid'].unique()[:2000]
        df_small = df[df['uid'].isin(users)]

        for uid, group in tqdm(df_small.groupby('uid'), desc="生成序列"):
            if len(group) < 3: continue
            all_q.append(group['iid'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 模型定义
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.config = config
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        if self.config.use_knowledge_concept:
            self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        if self.config.use_student_interaction:
            self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
            self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        q_feat = self.question_emb(question_ids)
        if self.config.use_knowledge_concept:
            k_emb_weight = self.concept_emb.weight
            q_k_agg = torch.matmul(q_k_relation, k_emb_weight)
            q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)
            q_feat = q_feat + q_k_feat
        if self.config.use_student_interaction:
            d_val = self.diff_values[question_ids].unsqueeze(-1)
            d_feat = self.diff_proj(d_val)
            q_feat = q_feat + d_feat
        return q_feat

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        z_perm = z.permute(0, 2, 1)
        conv_out = self.conv(z_perm)
        conv_out = conv_out[:, :, :z.size(1)].permute(0, 2, 1)
        out = self.norm(conv_out + residual)
        return self.dropout(self.relu(out))

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)
        h = x
        for layer in self.tcan_layers:
            h = layer(h)
        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

# ==========================================
# 4. 训练与运行
# ==========================================

def train_and_evaluate(config, q_seqs, r_seqs, s_seqs, q_k_rel, diff_values, exp_name):
    train_q, test_q, train_r, test_r, train_s, test_s = train_test_split(
        q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42
    )
    train_ds = AssistDataset(train_q, train_r, train_s, config.seq_len)
    test_ds = AssistDataset(test_q, test_r, test_s, config.seq_len)
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

    model = HIG_TCAN_CD(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    print(f"\n>>> 实验: [{exp_name}]")
    # 修改：初始化 best_results 增加 RMSE
    best_results = {"AUC": 0.0, "ACC": 0.0, "RMSE": 1.0}

    for epoch in range(config.epochs):
        model.train()
        for q, r, mask, s in train_loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h, q_emb, s_static = model(q, r, s)
            feat = torch.cat([h[:, :-1, :], q_emb[:, 1:, :], s_static[:, 1:, :]], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)
            loss = F.binary_cross_entropy(preds, r[:, 1:], reduction='none')
            loss = (loss * mask[:, 1:]).sum() / (mask[:, 1:].sum() + 1e-8)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for q, r, mask, s in test_loader:
                q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
                h, q_emb, s_static = model(q, r, s)
                preds = model.pred_mlp(torch.cat([h[:, :-1, :], q_emb[:, 1:, :], s_static[:, 1:, :]], dim=-1)).squeeze(-1)
                valid = mask[:, 1:].bool()
                if valid.sum() > 0:
                    all_preds.append(preds[valid].cpu().numpy())
                    all_targets.append(r[:, 1:][valid].cpu().numpy())

        if all_preds:
            y_p, y_t = np.concatenate(all_preds), np.concatenate(all_targets)
            auc = roc_auc_score(y_t, y_p)
            acc = accuracy_score(y_t, (y_p >= 0.5).astype(int))
            # 修改：计算 RMSE
            rmse = np.sqrt(mean_squared_error(y_t, y_p))

            if auc > best_results["AUC"]:
                best_results = {"AUC": auc, "ACC": acc, "RMSE": rmse}

            # 修改：打印信息增加 RMSE
            print(f"  Epoch {epoch+1}: AUC={auc:.4f}, ACC={acc:.4f}, RMSE={rmse:.4f}")

    return best_results

def main():
    config = Config()
    processor = DataProcessor(config)
    print("====== Junyi 消融实验准备 (修复版) ======")
    try:
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process()
    except Exception as e:
        print(f"数据加载失败: {e}")
        return

    experiments = [
        ("(1) Baseline (No Aggregation)", False, False),
        ("(2) Difficulty Interaction Only", True, False),
        ("(3) Knowledge Interaction Only", False, True),
        ("(4) Full HIG-TCAN Model", True, True)
    ]

    summary = []
    for name, si, kc in experiments:
        config.use_student_interaction = si
        config.use_knowledge_concept = kc
        res = train_and_evaluate(config, q_seqs, r_seqs, s_seqs, q_k_rel, diff_values, name)
        summary.append((name, res))

    print("\n====== 消融实验总结 (Summary) ======")
    # 修改：表格表头增加 RMSE
    print(f"{'Experiment Name':<35} | {'AUC':<8} | {'ACC':<8} | {'RMSE':<8}")
    print("-" * 70)
    for name, res in summary:
        print(f"{name:<35} | {res['AUC']:.4f}   | {res['ACC']:.4f}   | {res['RMSE']:.4f}")

if __name__ == "__main__":
    main()

====== Junyi 消融实验准备 (修复版) ======
正在准备下载/检查数据集: junyi
正在尝试自动安装 EduData 库...


downloader, INFO http://base.ustc.edu.cn/data/JunyiAcademy_Math_Practicing_Log/junyi.rar is saved as data/junyi.rar


downloader, INFO data/junyi.rar is unrar to data/junyi



读取 Junyi 数据: ./data/junyi/junyi_ProblemLog_for_PSLC.txt
统计: 学生 73244, 题目 2160


生成序列: 100%|██████████| 2000/2000 [00:00<00:00, 14305.68it/s]



>>> 实验: [(1) Baseline (No Aggregation)]
  Epoch 1: AUC=0.6423, ACC=0.6188, RMSE=0.4813
  Epoch 2: AUC=0.6853, ACC=0.6511, RMSE=0.4697
  Epoch 3: AUC=0.7219, ACC=0.6786, RMSE=0.4580
  Epoch 4: AUC=0.7547, ACC=0.7012, RMSE=0.4453
  Epoch 5: AUC=0.7815, ACC=0.7251, RMSE=0.4314

>>> 实验: [(2) Difficulty Interaction Only]
  Epoch 1: AUC=0.6956, ACC=0.6398, RMSE=0.4729
  Epoch 2: AUC=0.7303, ACC=0.6847, RMSE=0.4549
  Epoch 3: AUC=0.7541, ACC=0.6960, RMSE=0.4444
  Epoch 4: AUC=0.7760, ACC=0.7057, RMSE=0.4358
  Epoch 5: AUC=0.7951, ACC=0.7287, RMSE=0.4239

>>> 实验: [(3) Knowledge Interaction Only]
  Epoch 1: AUC=0.6298, ACC=0.6081, RMSE=0.4812
  Epoch 2: AUC=0.6849, ACC=0.6534, RMSE=0.4706
  Epoch 3: AUC=0.7063, ACC=0.6687, RMSE=0.4605
  Epoch 4: AUC=0.7347, ACC=0.6908, RMSE=0.4501
  Epoch 5: AUC=0.7638, ACC=0.7061, RMSE=0.4379

>>> 实验: [(4) Full HIG-TCAN Model]
  Epoch 1: AUC=0.6677, ACC=0.6295, RMSE=0.4767
  Epoch 2: AUC=0.7088, ACC=0.6608, RMSE=0.4622
  Epoch 3: AUC=0.7356, ACC=0.6875, RMSE=